In [14]:
import numpy as np

fname = "Lx_24_Nd_500_NT_17280_pz_0.0_MPI_hexagonal6.npz"

# npzファイルを開く
data = np.load(fname)

# キー一覧を表示
print("keys:", data.files)

# 各キーの shape や dtype を確認
for k in data.files:
    arr = data[k]
    print(f"{k}: shape={arr.shape}, dtype={arr.dtype}")

# 例えば pg をちょっと見る
print("pg[:10] =", data["pg"][:21])


keys: ['pg', 'TEE1_ave', 'TEE1_err', 'TEE1_var', 'TEE2_ave', 'TEE2_err', 'TEE2_var', 'TEE3_ave', 'TEE3_err', 'TEE3_var', 'TEE4_ave', 'TEE4_err', 'TEE4_var', 'TEE5_ave', 'TEE5_err', 'TEE5_var', 'TEE6_ave', 'TEE6_err', 'TEE6_var']
pg: shape=(21,), dtype=float64
TEE1_ave: shape=(21,), dtype=float64
TEE1_err: shape=(21,), dtype=float64
TEE1_var: shape=(21,), dtype=float64
TEE2_ave: shape=(21,), dtype=float64
TEE2_err: shape=(21,), dtype=float64
TEE2_var: shape=(21,), dtype=float64
TEE3_ave: shape=(21,), dtype=float64
TEE3_err: shape=(21,), dtype=float64
TEE3_var: shape=(21,), dtype=float64
TEE4_ave: shape=(21,), dtype=float64
TEE4_err: shape=(21,), dtype=float64
TEE4_var: shape=(21,), dtype=float64
TEE5_ave: shape=(21,), dtype=float64
TEE5_err: shape=(21,), dtype=float64
TEE5_var: shape=(21,), dtype=float64
TEE6_ave: shape=(21,), dtype=float64
TEE6_err: shape=(21,), dtype=float64
TEE6_var: shape=(21,), dtype=float64
pg[:10] = [0.7  0.71 0.72 0.73 0.74 0.75 0.76 0.77 0.78 0.79 0.8  0.81 0.8

In [17]:
import numpy as np
from collections import defaultdict
import re, math

# --- 1) slurmログをパースして {pg: list of [TEE1..TEE6]} を作る ---
bucket = defaultdict(list)
pat = re.compile(r"pg=\s*([0-9.]+).*?TEE1=\s*([-0-9.]+).*?TEE2=\s*([-0-9.]+).*?"
                 r"TEE3=\s*([-0-9.]+).*?TEE4=\s*([-0-9.]+).*?TEE5=\s*([-0-9.]+).*?TEE6=\s*([-0-9.]+)")
with open("slurm-2647209.out") as f:
    for line in f:
        m = pat.search(line)
        if m:
            pg = float(m.group(1))
            tees = list(map(float, m.groups()[1:]))  # 6個
            bucket[pg].append(tees)

pg = np.array(sorted(bucket.keys()))
stack = np.array([bucket[p] for p in pg], dtype=float)   # shape=(nP, nSamples, 6)

# --- 2) 分散とそのブートストラップ標準誤差を推定 ---
def bootstrap_var(x, nboot=1000, rng=np.random.default_rng(0)):
    x = np.asarray(x)
    n = len(x)
    if n <= 3:
        return np.var(x, ddof=1), np.nan  # サンプル少な過ぎ
    vs = []
    for _ in range(nboot):
        idx = rng.integers(0, n, n)
        vs.append(np.var(x[idx], ddof=1))
    return np.var(x, ddof=1), np.std(vs, ddof=1)

var6  = np.zeros((len(pg), 6))
dvar6 = np.zeros((len(pg), 6))
for i in range(len(pg)):
    for k in range(6):
        v, dv = bootstrap_var(stack[i,:,k], nboot=1000)
        var6[i,k]  = v
        dvar6[i,k] = dv

# これを npz に保存（pyfssa にそのまま渡せる形）
out = {"pg": pg}
for k in range(6):
    out[f"TEE{k+1}_var"] = var6[:,k]
    out[f"TEE{k+1}_err"] = dvar6[:,k]          # ← 平均の誤差ではなく、分散の標準誤差！
np.savez_compressed("from_log_var_boot.npz", **out)


In [18]:
import numpy as np

fname = "from_log_var_boot.npz"

# npzファイルを開く
data = np.load(fname)

# キー一覧を表示
print("keys:", data.files)

# 各キーの shape や dtype を確認
for k in data.files:
    arr = data[k]
    print(f"{k}: shape={arr.shape}, dtype={arr.dtype}")

# 例えば pg をちょっと見る
print("pg =", data["pg"][:21])
print("TEE2_var =", data["TEE2_var"][:21])
print("TEE2_err =", data["TEE2_err"][:21])
print("TEE3_var =", data["TEE3_var"][:21])
print("TEE3_err =", data["TEE3_err"][:21])
print("TEE4_var =", data["TEE4_var"][:21])
print("TEE4_err =", data["TEE4_err"][:21])

keys: ['pg', 'TEE1_var', 'TEE1_err', 'TEE2_var', 'TEE2_err', 'TEE3_var', 'TEE3_err', 'TEE4_var', 'TEE4_err', 'TEE5_var', 'TEE5_err', 'TEE6_var', 'TEE6_err']
pg: shape=(21,), dtype=float64
TEE1_var: shape=(21,), dtype=float64
TEE1_err: shape=(21,), dtype=float64
TEE2_var: shape=(21,), dtype=float64
TEE2_err: shape=(21,), dtype=float64
TEE3_var: shape=(21,), dtype=float64
TEE3_err: shape=(21,), dtype=float64
TEE4_var: shape=(21,), dtype=float64
TEE4_err: shape=(21,), dtype=float64
TEE5_var: shape=(21,), dtype=float64
TEE5_err: shape=(21,), dtype=float64
TEE6_var: shape=(21,), dtype=float64
TEE6_err: shape=(21,), dtype=float64
pg = [0.7  0.71 0.72 0.73 0.74 0.75 0.76 0.77 0.78 0.79 0.8  0.81 0.82 0.83
 0.84 0.85 0.86 0.87 0.88 0.89 0.9 ]
TEE2_var = [5.52001647e-04 1.97706549e-04 3.37018774e-04 8.27654810e-04
 1.49472392e-03 3.10639742e-03 4.64336418e-03 8.72635729e-03
 9.16490691e-03 7.25878317e-03 4.44266024e-03 2.14597683e-03
 9.35017631e-04 5.98849198e-04 3.15846188e-04 2.02102361e-04
